In [48]:
import pandas as pd

from pypdf import PdfReader
from pathlib import Path
import pdfplumber

In [49]:
data_folder = Path("../data/raw")

csv_files = list(data_folder.glob("*.csv"))
pdf_files = list(data_folder.glob("*.pdf"))

print("CSV files:")
for file in csv_files:
    print(file)

print("\nPDF files:")
for file in pdf_files:
    print(file)

CSV files:
..\data\raw\2022-COMMUNITY-PROJECTS-PETAUKE-CENTRAL.csv
..\data\raw\2025-NOT-APPROVED-COMMUNITY-PROJECTS-KAUMBWE2.csv
..\data\raw\2025-PROPOSED-COMMUNITY-PROJECTS-KAUMBWE2.csv

PDF files:
..\data\raw\11th-July-2025-Council-Minutes.pdf
..\data\raw\30th-April-2025- Council-Minutes.pdf
..\data\raw\Petauke-Town-Council-2025-OBB-Final-05.12.2024_Signed.pdf
..\data\raw\Petauke-Town-Council-2026-Budget-2026.pdf
..\data\raw\Petauke-Town-Council-Stratplan_2019-23.pdf
..\data\raw\Petauke.Lusangazi-Joint-IDP-Final.pdf


In [17]:
csv_data = {}

for file in csv_files:
    df = pd.read_csv(file)
    csv_data[file.name] = df

print(csv_data.keys())

dict_keys(['2022-COMMUNITY-PROJECTS-PETAUKE-CENTRAL.csv', '2025-NOT-APPROVED-COMMUNITY-PROJECTS-KAUMBWE2.csv', '2025-PROPOSED-COMMUNITY-PROJECTS-KAUMBWE2.csv'])


In [20]:
pdf_data = {}

for file in pdf_files:
    reader = PdfReader(file)

    text = ""

    for page in reader.pages:
        text += page.extract_text() or ""

    pdf_data[file.name] = text

print(pdf_data.keys())

dict_keys(['Petauke-Town-Council-2025-OBB-Final-05.12.2024_Signed.pdf', 'Petauke-Town-Council-2026-Budget-2026.pdf'])




INSPECTING "2026 BUDGET" AND "2025 OBB FINAL"

In [55]:
data_folder = Path("../data/raw") 
output_folder = Path("../data/processed") 
output_folder.mkdir(parents=True, exist_ok=True)

In [73]:
for pdf_file in pdf_files:

    if "2025-OBB" in pdf_file.name or "2026-Budget" in pdf_file.name:

        print(f"\nProcessing: {pdf_file.name}")

        extracted_tables = []

        with pdfplumber.open(pdf_file) as pdf:

            for page_num, page in enumerate(pdf.pages, start=1):

                tables = page.extract_tables()

                for table in tables:

                    if table:

                        df = pd.DataFrame(table)

                        # Remove empty rows and columns
                        df = df.fillna("")
                        df = df.map(
                            lambda cell: cell.strip()
                            if isinstance(cell, str)
                            else cell
                        )

                        df = df.loc[~(df == "").all(axis=1)]
                        df = df.loc[:, ~(df == "").all(axis=0)]

                        if not df.empty:
                            df["source_page"] = page_num
                            extracted_tables.append(df)

        # Save extracted tables
        if extracted_tables:

            final_df = pd.concat(
                extracted_tables,
                ignore_index=True
            )

            clean_name = (
                pdf_file.stem
                .lower()
                .replace("-", "_")
                .replace(".", "_")
            )

            output_file = (
                output_folder /
                f"db-unza26-csc4792-{clean_name}.csv"
            )

            final_df.to_csv(
                output_file,
                sep="|",
                index=False
            )

            print(f"Saved: {output_file}")
            print(f"Rows: {len(final_df)}")
            print(f"Columns: {len(final_df.columns)}")

        else:
            print("No tables found.")


Processing: Petauke-Town-Council-2025-OBB-Final-05.12.2024_Signed.pdf
Saved: ..\data\processed\db-unza26-csc4792-petauke_town_council_2025_obb_final_05_12_2024_signed.csv
Rows: 422
Columns: 9

Processing: Petauke-Town-Council-2026-Budget-2026.pdf
Saved: ..\data\processed\db-unza26-csc4792-petauke_town_council_2026_budget_2026.csv
Rows: 446
Columns: 9


In [ ]:
processed_files = list(output_folder.glob("*.csv"))

print("Processed CSV files:")

for file in processed_files:
    print(file)

In [ ]:
obb_dataset = pd.read_csv(
    "../data/processed/db-unza26-csc4792-petauke_town_council_2025_obb_final_05_12_2024_signed.csv",
    sep="|"
)

obb_dataset.head()

In [ ]:
budget_dataset = pd.read_csv(
    "../data/processed/db-unza26-csc4792-petauke_town_council_2026_budget_2026.csv",
    sep="|"
)

budget_dataset


In [68]:
obb_dataset.columns = obb_dataset.iloc[0]

obb_dataset = obb_dataset.iloc[1:].reset_index(drop=True)

In [ ]:
obb_dataset